# Matrix eigenvalues with native quantum phase estimation
Preserve the original matrix, seven signed phase bits and both evolution routes.
The two private data bits remain entangled after drop; they are not clean work.
Explicit local shot seeds replace the unseeded remote sampler.

## Request phase estimation with an explicit evolution contract

The visible roots prepare the data and compose QPE around exact unitary or
power-aware product-formula callbacks. The seven-bit signed fractional output
specifies phase interpretation. FlagQuantum supplies controlled powers, Fourier
operations and physical register assignment. Classical diagonalization supplies
the small exact unitary; approximate powers choose their declared repetitions.

Try a different normalized four-entry input. Phase peak weights should change
with its eigenvector overlaps while the spectral locations stay fixed. The
reported resolution interval is a discretization statement, not a sampling
confidence interval. Dropped data remains entangled physical storage; a zero
workspace budget does not mean the entire computation uses only seven bits.


In [ ]:
from io import BytesIO

from IPython.display import Image, display

import flagquantum as fq
from examples.synthesis.algorithms.quantum_phase_estimation.qpe_for_matrix.overview import (
    workflow as matrix_qpe,
)

print("Original matrix:", matrix_qpe.MATRIX)
print("Initial amplitudes:", matrix_qpe.INITIAL_AMPLITUDES)
print("Ordered Pauli decomposition:", matrix_qpe.PAULI_TERMS)
print("Normalization:", matrix_qpe.NORMALIZATION)
print("Phase bits / repetitions:", matrix_qpe.PHASE_BITS, matrix_qpe.REPETITIONS)


In [ ]:
import math
from typing import Annotated
from flagquantum import synth
from flagquantum.synth import (
    Output,
    QArray,
    QBit,
    QNum,
    allocate,
    drop,
    prepare_amplitudes,
    qfunc,
    qpe,
    suzuki_trotter,
    unitary,
)

UNITARY = matrix_qpe.UNITARY
NORMALIZED_OPERATOR = matrix_qpe.NORMALIZED_OPERATOR


@qfunc
def exact_phase(
    amplitudes: list[float], phase_result: Output[Annotated[QNum, (7, True, 7)]]
) -> None:
    state: QArray[QBit] = QArray()
    prepare_amplitudes(amplitudes, 0, state)
    allocate(phase_result)
    qpe(lambda: unitary(UNITARY, state), phase_result)
    drop(state)


@qfunc
def approximate_phase(
    amplitudes: list[float],
    repetitions: list[int],
    phase_result: Output[Annotated[QNum, (7, True, 7)]],
) -> None:
    state: QArray[QBit] = QArray()
    prepare_amplitudes(amplitudes, 0, state)
    allocate(phase_result)
    qpe(
        lambda power: suzuki_trotter(
            NORMALIZED_OPERATOR, -2 * math.pi * power, 2, repetitions[power], state
        ),
        phase_result,
    )
    drop(state)


repetitions = [1] * 65
for bit, count in enumerate(matrix_qpe.REPETITIONS):
    repetitions[1 << bit] = count
exact_program = synth.synthesize(
    exact_phase,
    classical_inputs={"amplitudes": list(matrix_qpe.INITIAL_AMPLITUDES)},
    max_qubits=9,
    max_workspace=0,
    memory_limit_bytes=64 * 1024**2,
)
approximate_program = synth.synthesize(
    approximate_phase,
    classical_inputs={
        "amplitudes": list(matrix_qpe.INITIAL_AMPLITUDES),
        "repetitions": repetitions,
    },
    max_qubits=9,
    max_workspace=0,
    memory_limit_bytes=64 * 1024**2,
)
print("Exact resources:", exact_program.report["resources"])
print("Approximate resources:", approximate_program.report["resources"])

print(fq.Circuit.from_ir(exact_program.circuit).draw())


In [ ]:
exact = matrix_qpe.run_estimate("exact", shots=10000, seed=1400)
approximate = matrix_qpe.run_estimate("approximate", shots=10000, seed=1401)
for estimate in (exact, approximate):
    print(estimate.route, "observed phase / energy / counts / probability")
    for row in estimate.outcomes:
        print(row)
    print("Maximum absolute state error:", estimate.maximum_state_error)


In [ ]:
for estimate in (exact, approximate):
    print(estimate.route, "top two energies:", estimate.solutions)
    print("Intervals:", estimate.intervals)
print("Energy resolution:", matrix_qpe.ENERGY_RESOLUTION)
print("All classical eigenvalues:", matrix_qpe.EIGENVALUES)


In [ ]:
figure = matrix_qpe.plot_estimates(exact, approximate)
with BytesIO() as image_buffer:
    figure.savefig(image_buffer, format="png", dpi=140)
    display(Image(data=image_buffer.getvalue()))
